# 02 · Text models: which sentiment model works best?

**Goal:** compare all sentiment models on the same test set and explain what they learned.

**Models:**
- *Ours, trained locally:* TextBlob (rules, course baseline), CountVectorizer + Naive Bayes, TF-IDF + Logistic Regression, MiniLM embeddings + Logistic Regression.
- *Ready-made from the internet, used as they are:* Cardiff RoBERTa (3 classes, trained on tweets), nlptown BERT (1–5 stars), the default Hugging Face `pipeline` model (positive/negative only).
- *Fine-tuned on our data (Colab GPU):* DistilBERT, and the Cardiff RoBERTa improved on our reviews.

**Inputs:** `data/processed/`, `results/metrics.json` (written by `scripts/train_text_models.py` and the Colab notebook).
**Outputs:** comparison figures in `results/figures/`.

Main metric: **macro-F1** (average of the three class F1 scores, so neutral counts as much as the others).
Every model picked its settings on the validation set; the test set was used once, at the end.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable from notebooks/

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1. Comparison table

In [ ]:
from src.evaluate import comparison_table, import_colab_metrics
print("Imported from Colab:", import_colab_metrics() or "nothing yet")
table = comparison_table("text")
table

In [ ]:
from src.evaluate import plot_model_comparison
plot_model_comparison(table, "test macro-F1");

**External test (Customer Feedback, 75 short reviews, positive vs negative only).** These are short, very
clear sentences, mostly not about products. It checks whether a model still works on unfamiliar text.

In [ ]:
plot_model_comparison(table, "external F1 (pos/neg)", filename="text_model_comparison_external");

## 2. Results per source (test set macro-F1)

In [ ]:
from src.evaluate import load_metrics
per_source = pd.DataFrame({name: {src: v["macro_f1"] for src, v in m["test_per_source"].items()}
                           for name, m in load_metrics()["text"].items()}).T
per_source.loc[table.index]

## 3. Confusion matrices: where do the models get confused?

In [ ]:
from pathlib import Path
figs = Path("../results/figures")
for name in table.index:
    f = figs / f"cm_text_{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=330))

## 4. Explainability: what does TF-IDF + Logistic Regression look for?

Logistic Regression gives every word (and word pair) a weight per class. The biggest weights are
the words that push a review towards that class. Word pairs like "not worth" or "stopped working"
are the reason we used 1–2 grams.

In [ ]:
import joblib
from src.config import path
from src.text_models import TfidfLogRegModel
from src.evaluate import plot_top_words
tfidf = TfidfLogRegModel()
tfidf.pipe = joblib.load(path("models_train") / "text_tfidf_logreg.joblib")
plot_top_words(tfidf.top_words(15));

## 5. Error analysis: the most confident mistakes

Reviews where TF-IDF + LR was *sure* and still wrong. These show the limits of a word-counting model and of star-based labels.

In [ ]:
from src.evaluate import misclassified_examples
test = pd.read_csv("../data/processed/text_test.csv")
misclassified_examples(test, "tfidf_logreg", n=10)

## 6. Issue tagger: *what* went wrong

Sentiment says how unhappy a customer is; the tagger (`src/issue_tags.py`) finds concrete problems
and shows the exact phrase that triggered each tag. It handles negation ("the box was **not** damaged" → no tag).

In [ ]:
from src.issue_tags import tag_issues
examples = [
    "The box arrived crushed and the jar inside leaked everywhere. I want a refund.",
    "Wrong size, not what I ordered. Returned it.",
    "Stopped working after a week, and the replacement never arrived.",
    "The bread was moldy and past its best before date.",
    "Box was not damaged at all, great packaging, fast delivery!",
]
pd.DataFrame({"review": examples, "tags found (phrase)": [tag_issues(e) for e in examples]})

There are no ground-truth tags, so as a sanity check we look at how often each tag fires per sentiment class.
A good tagger fires far more on negative reviews than on positive ones.

In [ ]:
from src.issue_tags import tag_rates
(tag_rates(test) * 100).round(1).rename(columns=lambda c: c + " %")

## Key findings

*(Written after all results are in, including the Colab fine-tunes.)*